# Feature Engineering

In this notebook, new features are created from the preprocessed ILINet dataset to support exploratory analysis and time-series forecasting. The engineered features capture temporal patterns and seasonal characteristics that are not explicitly available in the original dataset.

#### 1. Import Libraries

In [1]:
# Import libraries
import warnings

import numpy as np
import pandas as pd

from src.data_loader import load_processed_ilinet, PROCESSED_DATA

warnings.filterwarnings("ignore")

#### 2. Display Settings

In [2]:
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)

#### 3. Load Dataset

In [3]:
# Load processed dataset
df = load_processed_ilinet()

#### 4. Dataset Overview

In [4]:
# Display dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   REGION TYPE        1499 non-null   str           
 1   DATE               1499 non-null   datetime64[us]
 2   YEAR               1499 non-null   int64         
 3   WEEK               1499 non-null   int64         
 4   % WEIGHTED ILI     1499 non-null   float64       
 5   %UNWEIGHTED ILI    1499 non-null   float64       
 6   AGE 0-4            1499 non-null   int64         
 7   AGE 25-49          967 non-null    float64       
 8   AGE 25-64          627 non-null    float64       
 9   AGE 5-24           1499 non-null   int64         
 10  AGE 50-64          967 non-null    float64       
 11  AGE 65             1499 non-null   int64         
 12  ILITOTAL           1499 non-null   int64         
 13  NUM. OF PROVIDERS  1499 non-null   int64         
 14  TOTAL PATIENTS     

#### 5. Calendar Features

##### Month

In [5]:
# Extract month from date
df["month"] = df["DATE"].dt.month

# verify
df[["DATE", "month"]].head()

,DATE,month
0,1997-09-28,9
1,1997-10-05,10
2,1997-10-12,10
3,1997-10-19,10
4,1997-10-26,10


##### Month Name

In [6]:
# Extract month name from date
df["month_name"] = df["DATE"].dt.month_name()

# verify
df[["DATE", "month", "month_name"]].head()

,DATE,month,month_name
0,1997-09-28,9,September
1,1997-10-05,10,October
2,1997-10-12,10,October
3,1997-10-19,10,October
4,1997-10-26,10,October


##### Quarter

In [7]:
# Extract quarter from date
df["quarter"] = df["DATE"].dt.quarter

# verify
df[["DATE", "quarter"]].head()

,DATE,quarter
0,1997-09-28,3
1,1997-10-05,4
2,1997-10-12,4
3,1997-10-19,4
4,1997-10-26,4


#### 6. Seasonal Features

In [8]:
# Map months to meteorological seasons
month_to_season = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

df["season"] = df["month"].map(month_to_season)

In [9]:
# verify seasonal features
df[["DATE", "month", "season"]].head()

,DATE,month,season
0,1997-09-28,9,Autumn
1,1997-10-05,10,Autumn
2,1997-10-12,10,Autumn
3,1997-10-19,10,Autumn
4,1997-10-26,10,Autumn


In [10]:
# Check the distribution of seasons
df["season"].value_counts()

season
Spring    382
Autumn    374
Winter    373
Summer    370
Name: count, dtype: int64

#### 7. Lag Features

In [11]:
# Create lag features
df["weighted_ili_lag_1"] = df["% WEIGHTED ILI"].shift(1)
df["weighted_ili_lag_2"] = df["% WEIGHTED ILI"].shift(2)
df["weighted_ili_lag_4"] = df["% WEIGHTED ILI"].shift(4)
df["weighted_ili_lag_52"] = df["% WEIGHTED ILI"].shift(52)

In [12]:
# Display lag features
df[
    [
        "DATE",
        "% WEIGHTED ILI",
        "weighted_ili_lag_1",
        "weighted_ili_lag_2",
        "weighted_ili_lag_4",
        "weighted_ili_lag_52"
    ]
].head(55)

,DATE,% WEIGHTED ILI,weighted_ili_lag_1,weighted_ili_lag_2,weighted_ili_lag_4,weighted_ili_lag_52
0,1997-09-28,1.10148,NaN,NaN,NaN,NaN
1,1997-10-05,1.20007,1.10148,NaN,NaN,NaN
2,1997-10-12,1.37876,1.20007,1.10148,NaN,NaN
3,1997-10-19,1.19920,1.37876,1.20007,NaN,NaN
4,1997-10-26,1.65618,1.19920,1.37876,1.10148,NaN
...,...,...,...,...,...,...
50,1998-09-13,0.00000,0.00000,0.00000,0.00000,NaN
51,1998-09-20,0.00000,0.00000,0.00000,0.00000,NaN
52,1998-09-27,0.00000,0.00000,0.00000,0.00000,1.10148
53,1998-10-04,1.57733,0.00000,0.00000,0.00000,1.20007


In [13]:
# Check missing values in lag features
df[
    [
        "weighted_ili_lag_1",
        "weighted_ili_lag_2",
        "weighted_ili_lag_4",
        "weighted_ili_lag_52"
    ]
].isna().sum()

weighted_ili_lag_1      1
weighted_ili_lag_2      2
weighted_ili_lag_4      4
weighted_ili_lag_52    52
dtype: int64

#### 8. Rolling Features

In [14]:
# Calculate 4-week rolling mean
df["weighted_ili_rolling_mean_4"] = (
    df["% WEIGHTED ILI"]
    .shift(1).rolling(4).mean()
)

In [15]:
# Display rolling mean
df[
    [
        "DATE",
        "% WEIGHTED ILI",
        "weighted_ili_rolling_mean_4"
    ]
].head(10)

,DATE,% WEIGHTED ILI,weighted_ili_rolling_mean_4
0,1997-09-28,1.10148,NaN
1,1997-10-05,1.20007,NaN
2,1997-10-12,1.37876,NaN
3,1997-10-19,1.19920,NaN
4,1997-10-26,1.65618,1.219877
5,1997-11-02,1.41326,1.358553
6,1997-11-09,1.98680,1.411850
7,1997-11-16,2.44749,1.563860
8,1997-11-23,1.73901,1.875933
9,1997-11-30,1.93919,1.896640


In [16]:
# Calculate rolling statistics
df["weighted_ili_rolling_mean_12"] = (
    df["% WEIGHTED ILI"]
    .shift(1)
    .rolling(12)
    .mean()
)

df["weighted_ili_rolling_std_4"] = (
    df["% WEIGHTED ILI"]
    .shift(1)
    .rolling(4)
    .std()
)

#### 9. Cyclical Time Features

In [17]:
# Create cyclical week features
df["week_sin"] = np.sin(2 * np.pi * (df["WEEK"] - 1) / 52)
df["week_cos"] = np.cos(2 * np.pi * (df["WEEK"] - 1) / 52)

In [18]:
# Display cyclical week features
df[
    [
        "DATE",
        "WEEK",
        "week_sin",
        "week_cos"
    ]
].head(10)

,DATE,WEEK,week_sin,week_cos
0,1997-09-28,40,-1.000000,-1.836970e-16
1,1997-10-05,41,-0.992709,1.205367e-01
2,1997-10-12,42,-0.970942,2.393157e-01
3,1997-10-19,43,-0.935016,3.546049e-01
4,1997-10-26,44,-0.885456,4.647232e-01
5,1997-11-02,45,-0.822984,5.680647e-01
6,1997-11-09,46,-0.748511,6.631227e-01
7,1997-11-16,47,-0.663123,7.485107e-01
8,1997-11-23,48,-0.568065,8.229839e-01
9,1997-11-30,49,-0.464723,8.854560e-01


In [19]:
# Display engineered dataset columns
df.columns.tolist()

['REGION TYPE',
 'DATE',
 'YEAR',
 'WEEK',
 '% WEIGHTED ILI',
 '%UNWEIGHTED ILI',
 'AGE 0-4',
 'AGE 25-49',
 'AGE 25-64',
 'AGE 5-24',
 'AGE 50-64',
 'AGE 65',
 'ILITOTAL',
 'NUM. OF PROVIDERS',
 'TOTAL PATIENTS',
 'month',
 'month_name',
 'quarter',
 'season',
 'weighted_ili_lag_1',
 'weighted_ili_lag_2',
 'weighted_ili_lag_4',
 'weighted_ili_lag_52',
 'weighted_ili_rolling_mean_4',
 'weighted_ili_rolling_mean_12',
 'weighted_ili_rolling_std_4',
 'week_sin',
 'week_cos']

In [20]:
# Check missing values
df.isna().sum()

REGION TYPE                      0
DATE                             0
YEAR                             0
WEEK                             0
% WEIGHTED ILI                   0
                                ..
weighted_ili_rolling_mean_4      4
weighted_ili_rolling_mean_12    12
weighted_ili_rolling_std_4       4
week_sin                         0
week_cos                         0
Length: 28, dtype: int64

In [21]:
# Check feature data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   REGION TYPE                   1499 non-null   str           
 1   DATE                          1499 non-null   datetime64[us]
 2   YEAR                          1499 non-null   int64         
 3   WEEK                          1499 non-null   int64         
 4   % WEIGHTED ILI                1499 non-null   float64       
 5   %UNWEIGHTED ILI               1499 non-null   float64       
 6   AGE 0-4                       1499 non-null   int64         
 7   AGE 25-49                     967 non-null    float64       
 8   AGE 25-64                     627 non-null    float64       
 9   AGE 5-24                      1499 non-null   int64         
 10  AGE 50-64                     967 non-null    float64       
 11  AGE 65                        1499 non-nu

In [22]:
# Save engineered dataset
output_path = PROCESSED_DATA / "ILINet_engineered.csv"

df.to_csv(output_path, index=False)

## Summary

The preprocessed ILINet dataset was extended with features designed to capture temporal and seasonal patterns relevant to influenza forecasting.

The engineered features include calendar features (`month`, `month_name`, `quarter`, and `season`), lag features based on `% WEIGHTED ILI`, rolling statistics capturing recent activity and variability, and cyclical week features (`week_sin` and `week_cos`).

The final dataset contains 1,499 observations and 28 columns. The missing values introduced by lag and rolling features occur only where sufficient historical observations were not available.

The engineered dataset was saved as `ILINet_engineered.csv` in the `data/processed/` directory and is ready for the next stage of the forecasting workflow.